<a href="https://colab.research.google.com/github/OuhmadMohamed/DI_Bootcamp/blob/main/Week9/Day2/daily_chalange_w9_d2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to Finetune LLMs with LoRA


In [ ]:
%pip install peft==0.4.0
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset("Abirate/english_quotes", split="train[:10%]")

data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

train_sample = data.select(range(5))
display(train_sample)





/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

...

README.md: 0.00B [00:00, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Dataset({
    features: ['quote', 'author', 'tags', 'input_ids', 'attention_mask'],
    num_rows: 5
})

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

#Fill in `r=1` and `target_modules`.
lora_config = LoraConfig(
    r=1,
    lora_alpha= 1, # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules= ["query_key_value"],
    lora_dropout=0.1,
    bias="none", # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

#Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())


trainable params: 98,304 || all params: 559,312,896 || trainable%: 0.01757585078102687
None


In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

output_directory = os.path.join("../cache/working", "peft_lab_outputs")
training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate= 3e-2, # Higher learning rate than full fine-tuning.
    num_train_epochs=1,
    use_cpu=True,
    disable_tqdm=True, # Keep this to avoid any potential tqdm issues
    optim="adamw_torch" # Explicitly set optimizer to avoid fused kernel issues
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=data,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)
trainer.train()

In [ ]:

# Load the PEFT model using pre-defined LoRA configs and foundation model. We set `is_trainable=False` to avoid further training.

import time

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)


In [ ]:
#✅ Load model for inference##
from peft import PeftModel

peft_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False
)


In [ ]:

# Generate output tokens

inputs = tokenizer("Two things are infinite: ", return_tensors="pt")
outputs = peft_model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature = 0.8
    )

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))